# Finding Parkinson's disease taxonomic analyses

In this notebook we aim to imitate the analyses in ["`ABaCo` demo: Parkinson’s disease gut microbiome"](https://mona-abaco.readthedocs.io/en/latest/tutorial/demo-parkinson.html) where the aim was to "integrate the 9 studies while preserving key distinctions from the two patient states (Parkinson’s v.s. Healthy)."

```{margin}
After clicking the "Activate Notebook" button you can run the cells in this browser. Alternatively, you can also click on the 🚀 to launch in colab or binder.
```
<button title="Make live" style="display:inline-flex;align-items:center;gap:0.4rem;padding:0.5rem 1rem;border:0;border-radius:20px;background:linear-gradient(135deg,#0f766e,#14b8a6);color:white;cursor:pointer;font-size:1rem;" class="thebe-button" onclick="initThebeSBT()">Activate Notebook</button>

---

In [1]:
# uncomment if colab
# !pip install mgnipy

## Searching for studies using `MGnifier`

To start we configure our MGnipy client and access the MGnify API Studies resource.

We will filter our query to studies of the gut microbiome that mention "parkinson"s disease.

We can preview the resulting query urls via `.explain()`

In [1]:
from mgnipy import MGnipy

# Initialize MGnipy with a cache directory
MG = MGnipy(cache_dir="downloads")

# Search for studies related to Parkinson's disease in the human gut microbiome
pd_studies = MG.studies(
    search="parkinson",
    biome_lineage="root:Host-associated:Human:Digestive system:Large intestine:Fecal",
)

# Show all of the request urls for the search i.e., the query set
pd_studies.explain()

https://www.ebi.ac.uk/metagenomics/api/v2/studies?biome_lineage=root%3AHost-associated%3AHuman%3ADigestive+system%3ALarge+intestine%3AFecal&search=parkinson&page=1


looks good. we can proceed with actually executing the list query/queries via .get(). To enrich our list of studies with metadata details we can do this in bulk using `.enrich_details()` or asynchronously via `.aenrich_details()`

In [2]:
# as http client manager
with MG: 
    # populate study list
    pd_studies.get()
    # enrich study list with metadta
    await pd_studies.aenrich_details()

# can view as pandas or even save to file if you prefer
study_meta = pd_studies.detailed_metadata.to_pandas(expand_nested_dicts=True)

# taking a look 
study_meta

Enriching study details: 100%|██████████| 8/8 [00:00<00:00, 203.49it/s]


,accession,ena_accessions,title,updated_at,downloads,first_accession,metadata__study_name,metadata__center_name,metadata__study_title,metadata__study_accession,metadata__study_description,metadata__secondary_study_accession,biome__biome_name,biome__lineage
0,MGYS00001650,"[ERP004264, PRJEB4927]",Alterations of the Fecal Microbiome in Parkins...,2026-05-28T15:47:02.598000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP004264,Fecal Microbiome in Parkinson's Disease,Institute of Biotechnology;University of Helsi...,Alterations of the Fecal Microbiome in Parkins...,PRJEB4927,"In the course of Parkinson’s disease (PD), the...",ERP004264,Fecal,root:Host-associated:Human:Digestive system:La...
1,MGYS00006759,"[ERP146353, PRJEB61255]",EMG produced TPA metagenomics assembly of PRJN...,2026-05-28T15:47:02.660000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP146353,NaN,NaN,NaN,NaN,NaN,NaN,Fecal,root:Host-associated:Human:Digestive system:La...
2,MGYS00005755,"[PRJNA510730, SRP173877]",Microbiota composition of Parkinson's disease ...,2026-05-06T11:35:50.490000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",SRP173877,NaN,NaN,NaN,NaN,NaN,NaN,Fecal,root:Host-associated:Human:Digestive system:La...
3,MGYS00006760,"[ERP148661, PRJEB63522]",EMG produced TPA metagenomics assembly of PRJN...,2026-05-28T15:47:02.672000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP148661,NaN,NaN,NaN,NaN,NaN,NaN,Fecal,root:Host-associated:Human:Digestive system:La...
4,MGYS00005129,"[ERP109659, PRJEB27564]",Gut microbiota in Parkinson's disease: tempora...,2026-05-06T12:25:31.349000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP109659,Parkinson's disease gut microbiota follow-up,Institute of Biotechnology;University of Helsi...,Gut microbiota in Parkinson's disease: tempora...,PRJEB27564,Aiming to explore the temporal stability of gu...,ERP109659,Fecal,root:Host-associated:Human:Digestive system:La...
5,MGYS00006121,"[ERP142200, PRJEB57228]",Dietary intervention of people with Parkinson'...,2026-05-28T15:47:01.432000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP142200,NaN,NaN,NaN,NaN,NaN,NaN,Fecal,root:Host-associated:Human:Digestive system:La...
6,MGYS00005130,"[ERP112853, PRJEB30401]",Gut Microbiome Alterations Drive Distinct Meta...,2026-05-06T10:02:48.163000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP112853,Gut Microbiome and Parkinson's Disease,University of Cagliari,Gut Microbiome Alterations Drive Distinct Meta...,PRJEB30401,Parkinson's disease is a neurodegenerative dis...,ERP112853,Fecal,root:Host-associated:Human:Digestive system:La...
7,MGYS00005601,"[ERP113090, PRJEB30615]",Identification of Intestinal Bacterial Taxa wi...,2026-05-28T15:47:01.054000+00:00,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP113090,NaN,NaN,NaN,NaN,NaN,NaN,Fecal,root:Host-associated:Human:Digestive system:La...


Now that we found some studies that match our sesarch criteria, we can take a look at their datasets. 

---

## Using `MGazine` to access the study datasets

we can access the mgazine of datasets (kinda like a list of available datasets) via `.datasets` attribute. The study details we retrieved above will also be passed on to the mgazine

In [3]:
# access mgazine
MZ = pd_studies.datasets

# take a look
print(MZ)

MGazine containing:
- MGnify pipeline versions: ['v3', 'v4_1', 'v5', 'v6']
- Number of downloads: 72
- Short descriptions: ['Complete GO annotation',
 'DwC-Ready summary of 16S-V3-V4 ASV taxonomies using -PR2 as ref DB',
 'DwC-Ready summary of 16S-V3-V4 ASV taxonomies using -SILVA as ref DB',
 'DwC-Ready summary of closed-ref taxonomies using ITSoneDB as ref DB',
 'DwC-Ready summary of closed-ref taxonomies using PR2 as ref DB',
 'DwC-Ready summary of closed-ref taxonomies using SILVA-LSU as ref DB',
 'DwC-Ready summary of closed-ref taxonomies using SILVA-SSU as ref DB',
 'GO slim annotation',
 'InterPro matches',
 'Phylum level taxonomies',
 'Phylum level taxonomies LSU',
 'Phylum level taxonomies SSU',
 'Summary of DADA2-PR2 taxonomies',
 'Summary of DADA2-SILVA taxonomies',
 'Summary of ITSoneDB taxonomies',
 'Summary of PR2 taxonomies',
 'Summary of SILVA-LSU taxonomies',
 'Summary of SILVA-SSU taxonomies',
 'Taxonomic assignments',
 'Taxonomic assignments LSU',
 'Taxonomic assign

Notice in "Nonempty metadata sets" we can see that the study details we collected [above](#searching-for-studies-using-mgnifier) are preserved in the mgazine. 
```{toggle}
The `mgnify_studies` attribute is a `MGnifyMetadata` object so contains all the same methods for viewing e.g.: 
- `MZ_SSU.mgnify_studies.to_pandas(expand_nested_dicts=True)`
- `... .to_list()`
- `... .to_polars()`
- `... .records()`
- etc.

Later on in [Using MGnetizer to colllect more metadata](#using-mgnetizer-to-collect-more-metadata) we will assign additional sets of metadata to `.mgnify_runs` and `.biosamples_metadata` which will also convert the lists of records into a MGnifyMetadata object
```

### Filtering the dataset list

We can filter by the pipeline version and short descriptions of the datasets. 

For the ABaCo demo we will use the taxonomic analyses and we will use v4 onwards due to differences in pipeline versions and specifically SILVA databases that were used for the taxonomic analysis


In [4]:
# we can filter by passing as index 
V5 = MZ['v5']['Taxonomic assignments SSU']
V6 = MZ["Summary of SILVA-SSU taxonomies"]

print(V5, V6)

MGazine containing:
- MGnify pipeline versions: ['v5']
- Number of downloads: 4
- Short descriptions: ['Taxonomic assignments SSU']
- Nonempty metadata sets: .mgnify_studies
 MGazine containing:
- MGnify pipeline versions: ['v6']
- Number of downloads: 3
- Short descriptions: ['Summary of SILVA-SSU taxonomies']
- Nonempty metadata sets: .mgnify_studies



### Combining dataset lists

In [5]:
# can add magazines
MZ_SSU = V5 + V6

# print still works
print(MZ_SSU)

MGazine containing:
- MGnify pipeline versions: ['v5', 'v6']
- Number of downloads: 7
- Short descriptions: ['Summary of SILVA-SSU taxonomies', 'Taxonomic assignments SSU']
- Nonempty metadata sets: .mgnify_studies



Now that we have filtered our list of datasets a bit, let's actually get and merge the taxonomic datasets. 

---

### (Lazy)loading into one taxonomic dataset

Currently availble in mgnipy are special MGazines for handling taxonomic assignment datasets in the classic taxa x sample/run format `TaxaMGazine` or in Darwin core-ready format `DWCTaxaMGazine`

we can also access these from an existing MGazine instance via `.taxonomic` or `.taxonomic_dwc_ready`

In [ ]:
taxo = MZ_SSU.taxonomic

MGazine containing:
- MGnify pipeline versions: ['v5', 'v6']
- Number of downloads: 7
- Short descriptions: ['Summary of SILVA-SSU taxonomies', 'Taxonomic assignments SSU']
- Nonempty metadata sets: .mgnify_studies
-----------------------
Next steps: Use `.load()` to initialize.



now we can load the datasets as `polars.LazyFrame`

In [7]:
# lazyload the mgnify taxanomic assignments datasets
taxo.load()

# calling to_pandas or to_polars will collect the data and return a dataframe
taxo.to_pandas().head()

,taxonomy,ERZ17292930,ERZ17292940,ERZ17292950,ERZ17292931,ERZ17292941,ERZ17292951,ERZ17292932,ERZ17292942,ERZ17292952,...,ERR3006049,ERR3006050,ERR3006051,ERR3006052,ERR3006053,ERR3006054,ERR3006055,ERR3006056,ERR3006057,ERR3006058
0,sk__Archaea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,sk__Archaea;k__;p__Candidatus_Thermoplasmatota...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,sk__Archaea;k__;p__Candidatus_Thermoplasmatota...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,sk__Archaea;k__;p__Candidatus_Thermoplasmatota...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,sk__Archaea;k__;p__Candidatus_Thermoplasmatota...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


additionally there is a method `.taxonomic_metadata()` that parses "taxonomy" into the taxonomic ranks, returning as pandas or polars dataframe which is configured via arg `df_engine=`. The default is pandas.

In [12]:
# looking at taxonomic info
taxo.taxonomic_metadata().head()

'Summary of SILVA-SSU taxonomies' may be used for e.g., caching, `long_short_mapping`.


,Superkingdom,Kingdom,Phylum,Class,Order,Family,Genus,Species
0,Archaea,NA,NA,NA,NA,NA,NA,NA
1,Archaea,NA,Candidatus_Thermoplasmatota,Thermoplasmata,NA,NA,NA,NA
2,Archaea,NA,Candidatus_Thermoplasmatota,Thermoplasmata,Methanomassiliicoccales,NA,NA,NA
3,Archaea,NA,Candidatus_Thermoplasmatota,Thermoplasmata,Methanomassiliicoccales,Methanomassiliicoccaceae,Methanomassiliicoccus,NA
4,Archaea,NA,Candidatus_Thermoplasmatota,Thermoplasmata,Methanomassiliicoccales,Methanomassiliicoccaceae,Methanomassiliicoccus,Candidatus_Methanomassiliicoccus_intestinalis


also any run and sample metadata relevant to the observations (i.e., by run accessions) can be merged and viewed using `.metadata()` again as polars or pandas dataframes. As we know metadata() for the observations in the taxonomic mgazine is not available:

In [16]:
display(taxo.metadata().head())

print(taxo)

""
ERR2730148
ERR2730149
ERR2730150
ERR2730151
ERR2730152


MGazine containing:
- MGnify pipeline versions: ['v5', 'v6']
- Number of downloads: 7
- Short descriptions: ['Summary of SILVA-SSU taxonomies', 'Taxonomic assignments SSU']
- Nonempty metadata sets: .mgnify_studies



if we recall [from earlier](#using-mgazine-to-access-the-study-datasets) and again in the print statement, we only had `.mgnify_studies` enriched. 

We still need to enrich with run and/or sample metadata, which we will do next :) 

---

## Using `MGnetizer` to collect more metadata

PICK UP HERE
First from MGnify

In [17]:
runs_ids = [x for x in taxo.runs_accessions if x.startswith('ERR') or x.startswith('SRR')]
assembly_ids = [x for x in taxo.runs_accessions if x.startswith('ERZ')]

mnet_run = MG.mgnetizer(resource='run', all_ids=runs_ids)
mnet_acc = MG.mgnetizer(resource='assembly', all_ids=assembly_ids)
print(mnet_run, mnet_acc)

MGnetizer for resource 'run' with 876 ids. 
Enriched metadata contains 0 entries. 
 Detail proxy: type 
Cache directory: downloads/96fb45ac2e701602c8eceb572eabad7cf964eaf7b37bb6e91eef293d6aee6ee6 
 MGnetizer for resource 'assembly' with 952 ids. 
Enriched metadata contains 0 entries. 
 Detail proxy: type 
Cache directory: downloads/a76f9c0360700f77d1a0c026ae2262c86e68f91c83fb326a973a4991be729a69 



In [ ]:
# now making the 
with MG: 
    await mnet_run.aenrich(limit=None)
    await mnet_acc.aenrich(limit=None)

Enriching metadata from MGnify: 897it [00:00, ?it/s]
Enriching metadata from MGnify: 967it [00:00, ?it/s]


In [36]:
taxo.mgnify_runs = (mnet_run.metadata + mnet_acc.metadata).to_list(drop_duplicates=True)

### Bonus: Collecting even more metadata using `BioSampler`

we again recommend starting from your mgnipy instance to automatically pass on the configuration and for http client handling.

In [20]:
sample_ids = taxo.mgnify_runs.to_pandas()['sample_accession'].to_list()

bios = MG.biosampler(
    sample_ids=sample_ids
)

print(bios)

BioSampler with 1864 sample_ids. 
Enriched metadata contains 0 entries. 
Cache directory: downloads/cbc5243547de887e85ea5f6858fa2810ded35c4dad2c4a3dd3dd41d8037aab4f 



Note that the above is the same as calling

```python
from mgnipy.V2.collect import BioSampler

bios = BioSampler(
    sample_ids=sample_ids,
    config=MG.config,
    client=MG.client
)
```

In [21]:
with bios: 
    await bios.aenrich(limit=400, incl_ena=False)

Enriching biosamples: 100%|█████████▉| 1857/1864 [00:00<?, ?it/s]


In [22]:
taxo.biosamples_metadata = bios.metadata.to_list()
print(taxo)

MGazine containing:
- MGnify pipeline versions: ['v5', 'v6']
- Number of downloads: 7
- Short descriptions: ['Summary of SILVA-SSU taxonomies', 'Taxonomic assignments SSU']
- Nonempty metadata sets: .mgnify_runs, .mgnify_studies, .biosamples_metadata



## Enriching with metadata

### taxonomic info

if we take a look at the metadata it will be empty

In [24]:
taxo.metadata().merge(taxo.mgnify_studies.to_pandas(expand_nested_dicts=True), left_on='study_accession', right_on='accession', how='left').head()

,experiment_type,instrument_model,instrument_platform,sample_accession,study_accession,updated_at_x,run_accession,reads_study_accession,assembly_study_accession,assembler_name,...,downloads,first_accession,metadata__study_name,metadata__center_name,metadata__study_title,metadata__study_accession,metadata__study_description,metadata__secondary_study_accession,biome__biome_name,biome__lineage
0,Amplicon,NaN,NaN,SAMEA4821212,MGYS00005129,None,None,NaN,None,NaN,...,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP109659,Parkinson's disease gut microbiota follow-up,Institute of Biotechnology;University of Helsi...,Gut microbiota in Parkinson's disease: tempora...,PRJEB27564,Aiming to explore the temporal stability of gu...,ERP109659,Fecal,root:Host-associated:Human:Digestive system:La...
1,Amplicon,NaN,NaN,SAMEA4821212,MGYS00005129,None,None,NaN,None,NaN,...,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP109659,Parkinson's disease gut microbiota follow-up,Institute of Biotechnology;University of Helsi...,Gut microbiota in Parkinson's disease: tempora...,PRJEB27564,Aiming to explore the temporal stability of gu...,ERP109659,Fecal,root:Host-associated:Human:Digestive system:La...
2,Amplicon,NaN,NaN,SAMEA4821212,MGYS00005129,None,None,NaN,None,NaN,...,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP109659,Parkinson's disease gut microbiota follow-up,Institute of Biotechnology;University of Helsi...,Gut microbiota in Parkinson's disease: tempora...,PRJEB27564,Aiming to explore the temporal stability of gu...,ERP109659,Fecal,root:Host-associated:Human:Digestive system:La...
3,Amplicon,NaN,NaN,SAMEA4821212,MGYS00005129,None,None,NaN,None,NaN,...,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP109659,Parkinson's disease gut microbiota follow-up,Institute of Biotechnology;University of Helsi...,Gut microbiota in Parkinson's disease: tempora...,PRJEB27564,Aiming to explore the temporal stability of gu...,ERP109659,Fecal,root:Host-associated:Human:Digestive system:La...
4,Amplicon,NaN,NaN,SAMEA4821212,MGYS00005129,None,None,NaN,None,NaN,...,"[{'file_type': 'tsv', 'download_type': 'Taxono...",ERP109659,Parkinson's disease gut microbiota follow-up,Institute of Biotechnology;University of Helsi...,Gut microbiota in Parkinson's disease: tempora...,PRJEB27564,Aiming to explore the temporal stability of gu...,ERP109659,Fecal,root:Host-associated:Human:Digestive system:La...


In [26]:
taxo.metadata().isna().sum()

experiment_type         967
instrument_model       2106
instrument_platform    2106
sample_accession          0
study_accession         967
                       ... 
Radiation_Chemo        2106
SIBO                   2106
Sleep_aid              2106
Thyroid_med            2106
Ulcer_past_3_months    2106
Length: 164, dtype: int64

In [89]:
for col in taxo.metadata().columns:
    print(col)

experiment_type
instrument_model
instrument_platform
sample_accession
study_accession
updated_at
run_accession
reads_study_accession
assembly_study_accession
assembler_name
assembler_version
status
sample__accession
sample__ena_accessions
sample__sample_title
sample__biome
sample__updated_at
study__accession
study__ena_accessions
study__title
study__updated_at
study__metadata.study_name
study__metadata.center_name
study__metadata.study_title
study__metadata.study_accession
study__metadata.study_description
study__metadata.secondary_study_accession
study__biome.biome_name
study__biome.lineage
GivenID
SampleID
SRA accession
name
taxid
ENA first public
ENA-CHECKLIST
External Id
INSDC center name
INSDC last update
INSDC status
Submitter Id
broad-scale environmental context
collection date
description
environmental medium
geographic location (country and/or sea)
geographic location (latitude)
geographic location (longitude)
host age
host diet
host disease status
host family relationship
hos

In [97]:
taxo.mgnify_studies.to_list([{1: [2, 3]}])

[{1: [2, 3]}]

In [68]:
taxo.to_anndata()

'Summary of SILVA-SSU taxonomies' may be used for e.g., caching, `long_short_mapping`.
/Users/anglup/.pyenv/versions/3.11.7/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)
ERROR:mgnipy.V2.datasets.taxonomic:Returning without metadata() as var - Error occurred while converting to AnnData: Observations annot. `var` must have as many rows as `X` has columns (1828), but has 2106 rows.
'Summary of SILVA-SSU taxonomies' may be used for e.g., caching, `long_short_mapping`.
/Users/anglup/.pyenv/versions/3.11.7/lib/python3.11/functools.py:909: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


AnnData object with n_obs × n_vars = 2917 × 1828
    obs: 'Superkingdom', 'Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Genus', 'Species'

however we can add additional metadata that we collected manually, or taxacurator can help some

In [ ]:
# PICK UP HERE

In [14]:
from abaco.dataloader import DataPreprocess, one_hot_encoding

In [ ]:
# from abaco.dataloader import DataPreprocess, one_hot_encoding
# # Load Parkinson's disease dataset
# path_to_dataset = 'data/dataset_parkinson.csv'
# batch_col = "study_code"
# bio_col = "phenotype"
# id_col = "samples"

# # Convert data path into compatible pd.DataFrame
# df_parkinson = DataPreprocess(
#     path_to_dataset,
#     factors = [
#         id_col,
#         batch_col,
#         bio_col
#     ]
# ).dropna()

# # see if there are 3 categorical and n numeric columns (should be an extra column for location)
# print(df_parkinson.info())